In [1]:
from utils import get_private_key
from huggingface_hub import login
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM, BitsAndBytesConfig
from pprint import pprint


ModuleNotFoundError: No module named 'utils'

In [2]:
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM, BitsAndBytesConfig


/home/mrosaria/Projects/NLP/GymRat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-26 16:49:31.460260: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
model_id = "meta-llama/Llama-3.1-8B"


In [4]:
# do the  4-bit quantization configuration in Q-LORA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype='float16',
    bnb_4bit_use_double_quant=True
)

In [5]:
def _load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Ensure EOS token is set (usually already set)
    tokenizer.eos_token = tokenizer.eos_token or "</s>"
    # Set pad token to EOS if not defined
    #Using pad_token = eos_token is a common workaround for models like LLaMA.
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

    #left using commonly for generation and training, right for inference
    tokenizer.padding_size="right"
    
    # tokenizer.chat_template = {
    #     "system": "{input}",      # system instructions
    #     "user": "{input}",        # user input
    #     "assistant": "{output}"   # model output
    # }

#     tokenizer.chat_template = (
#     "{% for message in messages %}"
#     "{{ message['role'] | upper }}: {{ message['content'] }}\n"
#     "{% endfor %}"
#     "ASSISTANT:"
# )

    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{ message['role'] | upper }}: {{ message['content'] }}\n"
        "{% endfor %}"
        "ASSISTANT: Respond ONLY with a valid JSON object containing 'instruction' and 'output'. "
        "Do not include any extra text, explanation, or quotes.\n"
    )


    return tokenizer

def _load_model():
    return AutoModelForCausalLM.from_pretrained(model_id, 
                                                device_map="auto", 
                                                quantization_config=bnb_config
)

def create_pipeline():
    pipe = pipeline("text-generation",
                        model=_load_model(), 
                        tokenizer=_load_tokenizer(), 
                         dtype="auto", device_map="auto")
    return pipe

In [6]:
prompt = [{'role': 'system', 'content': 'You are a concise and helpful medical tutor. Based on the provided text, generate a JSON object with exactly ONE question (as \'instruction\') and ONE answer (as \'output\').\n\n- The content must relate to health, exercise, sports, fitness, or physiotherapy.\n- Do not include multiple questions or answers.\n- Do not repeat the instruction in the output.\n- The output must contain a thorough and detailed, multi-paragraph question (as \'instruction\') and answer (as \'output\').\n- If the text is not relevant, return: {"instruction": "NULL", "output": "NULL"}\n\n- Respond ONLY with the JSON object. Do NOT include any explanation or commentary.'}, {'role': 'user', 'content': "BODYWEIGHT SQUAT WITH BUTTWINK Now, if you are just performing a few bodyweight squats and butt winking occurs, it's likely not a big deal. Minimal power is generated at the spine during a normal-tempo air squat. However, as soon as you add a barbell, things change. If butt winking continues under load, the power generated at the spine increases at one or two specic joints of the lumbar spine (usually L4/5 and L5/S1). Therefore, when you have a stress"}]
chat = create_pipeline()

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:34<00:00,  8.51s/it]
Device set to use cuda:0


In [ ]:
# tokenizer = _load_tokenizer()
# tokenizer.chat_template

In [ ]:
# Use the tokenizer’s built-in chat template rendering
# # formatted_prompt = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)

In [7]:
# chat_completion = chat(formatted_prompt,
chat_completion = chat(prompt,
                       #template=tokenizer.chat_template,
                       max_length=384,
                        do_sample=True,
                        temperature=0.75,
                        top_p=0.65,
                        num_return_sequences=1)

Both `max_new_tokens` (=256) and `max_length`(=384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [9]:
from pprint import pprint

In [10]:
pprint(chat_completion[0]['generated_text'])

[{'content': 'You are a concise and helpful medical tutor. Based on the '
             'provided text, generate a JSON object with exactly ONE question '
             "(as 'instruction') and ONE answer (as 'output').\n"
             '\n'
             '- The content must relate to health, exercise, sports, fitness, '
             'or physiotherapy.\n'
             '- Do not include multiple questions or answers.\n'
             '- Do not repeat the instruction in the output.\n'
             '- The output must contain a thorough and detailed, '
             "multi-paragraph question (as 'instruction') and answer (as "
             "'output').\n"
             '- If the text is not relevant, return: {"instruction": "NULL", '
             '"output": "NULL"}\n'
             '\n'
             '- Respond ONLY with the JSON object. Do NOT include any '
             'explanation or commentary.',
  'role': 'system'},
 {'content': 'BODYWEIGHT SQUAT WITH BUTTWINK Now, if you are just performing a '

In [ ]:
vhv vhn

SyntaxError: invalid syntax (3828512004.py, line 1)

In [ ]:
pprint(chat_completion[0]['generated_text'])

[{'content': 'You are a concise and helpful medical tutor. Based on the '
             'provided text, generate a JSON object with exactly ONE question '
             "(as 'instruction') and ONE answer (as 'output').\n"
             '\n'
             '- The content must relate to health, exercise, sports, fitness, '
             'or physiotherapy.\n'
             '- Do not include multiple questions or answers.\n'
             '- Do not repeat the instruction in the output.\n'
             '- The output must contain a thorough and detailed, '
             "multi-paragraph question (as 'instruction') and answer (as "
             "'output').\n"
             '- If the text is not relevant, return: {"instruction": "NULL", '
             '"output": "NULL"}\n'
             '\n'
             '- Respond ONLY with the JSON object. Do NOT include any '
             'explanation or commentary.',
  'role': 'system'},
 {'content': 'BODYWEIGHT SQUAT WITH BUTTWINK Now, if you are just performing a '

In [ ]:
pprint(chat_completion[0]['generated_text'][-1]['content']['instruction'])

TypeError: string indices must be integers, not 'str'

In [ ]:
pprint(chat_completion[0]['generated_text'][2]['content']['output'])

TypeError: string indices must be integers, not 'str'

In [ ]:
from synthetic_generator import SyntheticDataGenerator

In [ ]:
file_name = "../../data/processed/rebuilding_milo_chunks_docling_max_tokens128_min_tokens50_meta_llama3p18B.txt" 


In [ ]:
data_gen = SyntheticDataGenerator(file_name)

In [ ]:
data_gen.get_samples()

AttributeError: 'SyntheticDataGenerator' object has no attribute 'get_samples'